# Tabular — โจทย์แบบ "ทำนายคลาสจากตาราง" (Classification)

ข้อมูลเป็นตาราง (แต่ละแถว = 1 ตัวอย่าง, คอลัมน์ = ฟีเจอร์) -> ทำนายป้าย (เช่น ป่วย/ไม่ป่วย, ซื้อ/ไม่ซื้อ)

วิธีในโน้ตบุ๊กนี้:
- วิธีที่ 1 (ง่ายสุด มือใหม่ทำแค่นี้) = `AutoGluon` กดรันแล้วมันลองหลายโมเดล + รวมให้เอง
- วิธีที่ 2 (ไม่บังคับ) = `LightGBM` + cross-validation คุมเอง


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ: ข้อมูลเป็นตาราง และทำนาย "ป้าย/หมวด" (มีกี่หมวดก็ได้)
ถ้าทำนาย "ตัวเลขต่อเนื่อง" (ราคา/คะแนน) -> ไปใช้ `regression.ipynb`

ต้องแก้ (`# << แก้`): ชื่อ competition, ชื่อไฟล์ csv, `TARGET` (คอลัมน์ที่ทำนาย), `METRIC`

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install -U "autogluon.tabular[all]"      # วิธีที่ 1
!pip -q install lightgbm scikit-learn pandas numpy   # วิธีที่ 2

## ขั้นที่ 2 — ตั้งค่า Kaggle แล้วดาวน์โหลดข้อมูล

ต้องมีไฟล์ `kaggle.json` ก่อน วิธีได้มา: เข้า kaggle.com -> กดรูปโปรไฟล์ -> Settings -> Account -> Create New Token
จะได้ไฟล์ `kaggle.json` หน้าตา `{"username":"...","key":"..."}`

- ถ้ารันบน `Kaggle Notebook`: ข้อมูลอยู่ที่ `/kaggle/input/...` แล้ว ไม่ต้องทำอะไร รันผ่านได้เลย
- ถ้ารันบน `Google Colab / เครื่องตัวเอง`: เอา username กับ key ใส่ในเซลล์ข้างล่าง (จุด `# << แก้`)

In [ ]:
import os, glob, subprocess

COMP = "super-ai-engineer-ss-6-heart-disease-prediction"      # << แก้ตรงนี้: slug ของ competition คือคำท้าย URL เช่น kaggle.com/competitions/titanic -> ใส่ "titanic"
KAGGLE_USERNAME = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) เช่น "0a1b2c3d4e5f6a7b8c9d..." (คีย์ยาว ~32 ตัวจาก kaggle.json)

def get_data(comp):
    # ถ้าอยู่บน Kaggle ข้อมูลถูกต่อไว้ให้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle แล้ว ข้อมูลอยู่ที่", kpath); return kpath
    # ถ้าไม่ใช่ Kaggle: เขียน token แล้วโหลดด้วย kaggle CLI
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("ไฟล์ที่ดาวน์โหลดมา (ดูชื่อไฟล์/โฟลเดอร์ แล้วเอาไปแก้ในเซลล์ถัดไป):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — โหลดข้อมูล + CONFIG

In [ ]:
import os, pandas as pd, numpy as np
TRAIN_CSV  = os.path.join(DATA_DIR,"train.csv")   # << แก้ชื่อไฟล์
TEST_CSV   = os.path.join(DATA_DIR,"test.csv")
SAMPLE_SUB = os.path.join(DATA_DIR,"sample_submission.csv")
train=pd.read_csv(TRAIN_CSV); test=pd.read_csv(TEST_CSV); sample=pd.read_csv(SAMPLE_SUB)
ID_COL = sample.columns[0]    # เดาอัตโนมัติ: คอลัมน์แรกของ sample_submission คือ id
TARGET = sample.columns[1]    # << แก้ถ้าผิด: คอลัมน์ที่ต้องทำนาย เช่น "target", "Survived", "num" (ปกติคือคอลัมน์ที่ 2 ของ sample)
# << แก้ METRIC ให้ตรง tab Evaluation บน Kaggle: ถ้าเขียน Accuracy ใส่ "accuracy", ROC AUC ใส่ "roc_auc", F1 ใส่ "f1"
METRIC = "accuracy"
print("คอลัมน์ train:", list(train.columns)); display(train.head()); display(sample.head())
print("เป้าหมาย =", TARGET, "| metric =", METRIC)

## วิธีที่ 1 — AutoGluon (ง่ายสุด มือใหม่ทำแค่นี้พอ)

AutoGluon จัดการ missing/categorical ให้เอง + ลองหลายโมเดล + รวมเป็น ensemble แค่บอก label กับ metric

In [ ]:
from autogluon.tabular import TabularPredictor
# แปลงชื่อ metric -> ชื่อของ AutoGluon (ถ้าไม่รู้จัก ส่งตรง ๆ ให้ AutoGluon ตัดสิน)
AG_METRIC = {"accuracy":"accuracy","acc":"accuracy","roc_auc":"roc_auc","auc":"roc_auc","f1":"f1"}.get(METRIC, METRIC)
train_ag = train.drop(columns=[c for c in [ID_COL] if c in train.columns])
test_ag  = test.drop(columns=[c for c in [ID_COL] if c in test.columns])
predictor = TabularPredictor(label=TARGET, eval_metric=AG_METRIC, path="ag_tab").fit(
    train_ag, presets="best_quality", time_limit=600)   # << แก้ time_limit: วินาที (600=10นาที) ลอง 120 ก่อน, ส่งจริงเพิ่มเป็น 1800
out = sample.copy()
if AG_METRIC == "roc_auc":
    # AUC ส่ง "ความน่าจะเป็นของคลาสบวก" -- ใช้ positive_class ของ AutoGluon ปลอดภัยกว่า iloc[:,1]
    out[TARGET] = predictor.predict_proba(test_ag)[predictor.positive_class].values
else:
    out[TARGET] = predictor.predict(test_ag).values                  # acc/f1 ส่งป้าย (เลข/ข้อความ)
out.to_csv("submission.csv", index=False)
print("บันทึก submission.csv"); display(out.head())
print(predictor.leaderboard().head())   # AutoGluon รุ่นใหม่เอา silent= ออกแล้ว

## วิธีที่ 2 — LightGBM + cross-validation (ไม่บังคับ)

คุมเองได้ละเอียด ใช้ StratifiedKFold กัน overfit

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
y = train[TARGET].values
X = train.drop(columns=[c for c in [ID_COL,TARGET] if c in train.columns])
Xte = test.drop(columns=[c for c in [ID_COL] if c in test.columns])
for df in (X,Xte):
    for col in df.columns:
        if df[col].dtype=="object": df[col]=df[col].astype("category")
oof=np.zeros(len(y)); pred=np.zeros(len(Xte))
for tr,va in StratifiedKFold(5,shuffle=True,random_state=42).split(X,y):
    m=lgb.LGBMClassifier(n_estimators=1500,learning_rate=0.02,num_leaves=31,random_state=42,verbose=-1)
    m.fit(X.iloc[tr],y[tr],eval_set=[(X.iloc[va],y[va])],callbacks=[lgb.early_stopping(80,verbose=False)])
    oof[va]=m.predict_proba(X.iloc[va])[:,1]; pred+=m.predict_proba(Xte)[:,1]/5
print("OOF AUC =", round(roc_auc_score(y,oof),4), "(ยิ่งใกล้ 1 ยิ่งดี, 0.5 = เดามั่ว)")
# pred = ความน่าจะเป็นว่าเป็นคลาส 1 ; เลือกแบบส่งให้ตรง metric อัตโนมัติ
out=sample.copy()
if METRIC in ("roc_auc","auc"):
    out[TARGET]=pred                      # AUC -> ส่งความน่าจะเป็น (ทศนิยม 0..1)
else:
    out[TARGET]=(pred>=0.5).astype(int)   # accuracy/f1 -> ส่งป้าย 0/1 (ตัดที่ 0.5)
out.to_csv("submission_lgbm.csv",index=False); print("บันทึก submission_lgbm.csv")

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
FILE = "submission.csv"   # << แก้เป็นชื่อไฟล์ที่คะแนนดีสุด
!kaggle competitions submit -c {COMP} -f {FILE} -m "autogluon tabular classification"
!kaggle competitions submissions -c {COMP} | head